# Demo — from `demo/config.yaml` to the final document

Input is a YAML file, not settings in this notebook — edit
`demo/config.yaml` (model, extractor, sample range) and re-run this
notebook top to bottom. Nothing here needs to change.


## Input — demo/config.yaml, as written on disk


In [3]:

import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

import dmpbridge
print(f'dmpbridge {dmpbridge.__version__}, installed at {Path(dmpbridge.__file__).parent}')

from dmpbridge.core import paths as P

CONFIG_PATH = Path('demo/config.yaml')
print(CONFIG_PATH.read_text(encoding='utf-8'))


dmpbridge 0.1.0, installed at c:\Users\Nahid\dmpbridge\dmpbridge
# Edit this file, then run:
#   python scripts/run_demo.py
#
# Results (the final labeled DMP document, one JSON per sample) are written
# into demo/output/ when it finishes.

name: Demo run
strategy: wholedoc
provider: ollama
host: http://localhost:11434

model: llama3.1:8b          # any model already pulled in Ollama
extractor: pdfplumber       # pdfplumber | docling | lighton

pdf_dir: data/input/pdfs    # expects sample1.pdf, sample2.pdf, ...
sample_start: 1
sample_end: 3                # keep this small for a quick demo run



## Run it


In [2]:
cfg = ExperimentConfig.from_yaml(CONFIG_PATH)
exp = Experiment(cfg)
exp.run()

print(f'\n{cfg.name}: {len(cfg.models)} model(s), {len(cfg.extractors)} extractor(s), '
      f'samples {cfg.sample_start}-{cfg.sample_end}')


NameError: name 'ExperimentConfig' is not defined

## Output — the final document

Same content `scripts/run_demo.py` copies into `demo/output/final/`; read here
directly from the standard pipeline location so this always reflects the latest run.


In [ ]:
model, extractor = cfg.models[0], cfg.extractors[0]
tag = cfg.tag_for(model, extractor)

for n in cfg.sample_range:
    final = P.final_path(tag, n)
    if not final.exists():
        continue
    doc = json.loads(final.read_text(encoding='utf-8'))
    template = doc['narrative']['template']

    print(f'=== sample{n} ===')
    print(f'TITLE: {template["title"]}\n')
    for i, section in enumerate(template['section'], 1):
        print(f'{i}. {section["title"]}')
        for q in section['question']:
            answer = q['answer']['json']['answer']
            print(f'   Q: {q["text"][:70]}')
            print(f'   A: {answer[:90]}{"..." if len(answer) > 90 else ""}')
    print()


=== sample1 ===
TITLE: DATA MANAGEMENT AND SHARING PLAN

1. Element 1: Data Type:
   Q: A. Types and amount of scientific data expected to be generated in the
   A: This secondary data analysis project will analyze deidentified data from 48,218 participan...
2. B. Scientific data that will be preserved and shared, and the rationale for doing so:
   Q: B. Scientific data that will be preserved and shared, and the rational
   A: As this is a secondary data analysis project, we will only be able to publicly share in th...
3. C. Metadata, other relevant data, and associated documentation:
   Q: C. Metadata, other relevant data, and associated documentation:
   A: In addition to the data described above, code and models will be included in the repositor...
4. Element 2: Related Tools, Software and/or Code:
   Q: Element 2: Related Tools, Software and/or Code:
   A: Data will be analyzed with custom code by our statistical and computer science team. ActiG...
5. Element 3: Standards:
   Q: Th